<a href="https://colab.research.google.com/github/DivyamAwasthy/Paper-Insight/blob/main/paper_insight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q sentence-transformers feedparser

import feedparser
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


def fetch_arxiv(query: str = "cat:cs.LG", max_results: int = 30):
    """Fetch (title, abstract) pairs from arXiv for a given search query."""
    url = (
        "http://export.arxiv.org/api/query?"
        f"search_query={query}&start=0&max_results={max_results}"
        "&sortBy=submittedDate&sortOrder=descending"
    )
    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        papers.append({
            "title": entry.title.strip().replace("\n", " "),
            "abstract": entry.summary.strip().replace("\n", " "),
        })
    return papers


MODEL_NAME = "all-MiniLM-L6-v2"


def embed_texts(model, texts):
    return model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def search(model, query, corpus_embeddings, papers, top_k=5):
    query_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(query_emb, corpus_embeddings)[0]
    ranked = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in ranked:
        results.append({
            "score": float(sims[idx]),
            "title": papers[idx]["title"],
            "abstract": papers[idx]["abstract"],
        })
    return results

print("Functions loaded: fetch_arxiv, embed_texts, search")

Functions loaded: fetch_arxiv, embed_texts, search


In [15]:
papers = fetch_arxiv(query="all:graph+neural+network", max_results=100)
print(f"got {len(papers)} papers")

model = SentenceTransformer(MODEL_NAME)
corpus_embeddings = embed_texts(model, [p["abstract"] for p in papers])
print("embeddings shape:", corpus_embeddings.shape)   # expect (100, 384)

got 100 papers


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

embeddings shape: (100, 384)


In [16]:
hits = search(model, "graph neural networks for molecules", corpus_embeddings, papers, top_k=5)
for i, h in enumerate(hits, 1):
    print(f"\n[{i}] score={h['score']:.3f}")
    print("   ", h["title"])
    print("   ", h["abstract"][:200], "...")


[1] score=0.381
    Software package MaRDI Open Interfaces for improved interoperability in numerical optimization
    To address the challenges of interoperability in computational science, we present the latest updates to the software package MaRDI Open Interfaces. This software package aims to decrease the time and ...

[2] score=0.380
    A Defect-Free Model of Amorphous Silicon with Pristine Electronic Structure
    Amorphous silicon (a-Si) is understood to be the canonical continuous random network material, ideally defined by fully fourfold coordination. Here, we show that a defect-free ('ideal') model of a-Si  ...

[3] score=0.378
    UCLCHEM 4.0: An open source gas-grain astrochemistry simulation framework
    Astrochemical modeling is a key tool for the understanding of the formation and destruction of molecules in the dense gas of the interstellar medium, as observed by modern day observational facilities ...

[4] score=0.377
    Geometry-Aware Superpixel Graph Transformer 

In [17]:
papers = fetch_arxiv(query="all:graph+neural+network", max_results=100)
print(f"got {len(papers)} papers")

model = SentenceTransformer(MODEL_NAME)
corpus_embeddings = embed_texts(model, [p["abstract"] for p in papers])
print("embeddings shape:", corpus_embeddings.shape)   # expect (30, 384)

got 100 papers


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

embeddings shape: (100, 384)


In [18]:
# ============================================================
# STAGE 2 — Structured extraction via the Gemini API (free tier)
# ============================================================
# Stage 1 found relevant papers. Stage 2 *analyzes* one: given a paper's
# abstract, ask an LLM to pull out structured fields (method / results /
# limitations) as JSON we can use programmatically.
#
# DEFEND THIS: Stage 1 (embeddings) is RETRIEVAL — find relevant text.
# Stage 2 (LLM) is EXTRACTION/REASONING — understand and structure that text.
# Two different jobs. The architecture is model-agnostic: we swapped the LLM
# provider (Anthropic -> Gemini) by changing only this function's internals.

!pip install -q google-generativeai

import json
from google.colab import userdata
import google.generativeai as genai

# Pull the key from Colab Secrets (never hard-code it)
genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

EXTRACTION_MODEL = "gemini-2.5-flash-lite"
model_llm = genai.GenerativeModel(EXTRACTION_MODEL)


def extract_paper_info(abstract: str) -> dict:
    """Ask the LLM to extract structured fields from a paper abstract.

    Returns a dict with keys: method, results, limitations.
    We force JSON output so the result is machine-usable, not prose."""

    # DEFEND THIS: the prompt is the 'program'. We (1) give the model a clear
    # role, (2) specify exactly the fields we want, (3) demand JSON-only output
    # so we can parse it. Prompt design IS the engineering here.
    prompt = f"""You are a research assistant analyzing a scientific paper abstract.
Extract the following from the abstract below. If a field is not stated, write "not stated".

Return ONLY a valid JSON object with exactly these keys:
- "method": the main approach or technique used (1-2 sentences)
- "results": the key findings or outcomes (1-2 sentences)
- "limitations": any stated limitations or caveats (1-2 sentences)

Do not include any text outside the JSON object.

ABSTRACT:
{abstract}"""

    response = model_llm.generate_content(prompt)
    raw = response.text.strip()

    # DEFEND THIS: LLMs sometimes wrap JSON in ```json fences. We strip those
    # before parsing so json.loads doesn't choke. Defensive parsing.
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"method": "PARSE_ERROR", "results": raw, "limitations": ""}


print("Stage 2 (Gemini) ready: extract_paper_info()")

Stage 2 (Gemini) ready: extract_paper_info()


In [19]:
import google.generativeai as genai
from google.colab import userdata
genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [20]:
# Test extraction on the top search hit from Stage 1
test_paper = papers[0]   # or pick any: papers[5], etc.
print("PAPER:", test_paper["title"], "\n")

info = extract_paper_info(test_paper["abstract"])
print("METHOD:     ", info["method"])
print("RESULTS:    ", info["results"])
print("LIMITATIONS:", info["limitations"])

PAPER: Structuring and Tokenizing Distributed User Interest Context for Generative Recommendation 

METHOD:      G2Rec is a scalable framework that unifies holistic graph-based user co-engagement modeling with semantic tokenization for industrial-scale generative recommendation. It captures holistic and semantically grounded user interest prototypes without requiring ground-truth user interests.
RESULTS:     G2Rec provides more comprehensive and accurate modeling of user behavior contexts in industrial sequential recommendation. Online deployment and extensive experiments demonstrate its superiority over existing methods.
LIMITATIONS: not stated


In [21]:
# ============================================================
# STAGE 3 — Grounded Q&A (Retrieval-Augmented Generation)
# ============================================================
# This COMBINES Stage 1 + Stage 2:
#   1. RETRIEVE: use the encoder + cosine search to find the most relevant
#      papers for a question (Stage 1's search()).
#   2. GENERATE: feed those papers to the LLM and ask it to answer the
#      question USING ONLY those papers, with citations.
#
# DEFEND THIS: this is RAG. The LLM doesn't answer from its own memory — we
# *ground* it in retrieved documents. That's what makes the answer (a) about
# OUR corpus and (b) citable/verifiable, instead of a generic or hallucinated
# response. Retrieval finds the evidence; generation reasons over it.

def answer_question(question: str, top_k: int = 3) -> dict:
    """Answer a question grounded in the most relevant retrieved papers.

    Returns a dict with the answer text and the papers used as sources."""

    # STEP 1 — RETRIEVE: reuse Stage 1's search to get the top-k relevant papers
    hits = search(model, question, corpus_embeddings, papers, top_k=top_k)

    # STEP 2 — Build a context block from the retrieved papers.
    # DEFEND THIS: we number the sources so the LLM can cite them ([1], [2]...).
    context = ""
    for i, h in enumerate(hits, 1):
        context += f"[{i}] {h['title']}\n{h['abstract']}\n\n"

    # STEP 3 — GENERATE: ask the LLM to answer using ONLY the retrieved context.
    # DEFEND THIS: the instruction to use only the provided papers and to say
    # when the answer isn't found is what prevents hallucination — the model is
    # constrained to the evidence we gave it.
    prompt = f"""You are a research assistant. Answer the question using ONLY the papers provided below.
Cite the papers you use with their numbers, e.g. [1], [2].
If the papers do not contain the answer, say "The retrieved papers do not address this question."

PAPERS:
{context}
QUESTION: {question}

Answer (with citations):"""

    response = model_llm.generate_content(prompt)
    answer = response.text.strip()

    return {
        "question": question,
        "answer": answer,
        "sources": [h["title"] for h in hits],
    }


print("Stage 3 ready: answer_question()")

Stage 3 ready: answer_question()


In [22]:
result = answer_question("What methods are used for graph neural networks on molecules?")
print("Q:", result["question"], "\n")
print("ANSWER:\n", result["answer"], "\n")
print("SOURCES USED:")
for s in result["sources"]:
    print("  -", s)

Q: What methods are used for graph neural networks on molecules? 

ANSWER:
 The retrieved papers do not address this question. 

SOURCES USED:
  - Software package MaRDI Open Interfaces for improved interoperability in numerical optimization
  - Boundary Embedding Shaping with Adaptive Contrastive Learning for Graph Structural Disentanglement
  - Quantization of Brane-Skyrmions via Physics-Informed Neural Networks


In [23]:
result = answer_question("What approaches use physics-informed neural networks?")
print("Q:", result["question"], "\n")
print("ANSWER:\n", result["answer"], "\n")
print("SOURCES USED:")
for s in result["sources"]:
    print("  -", s)

Q: What approaches use physics-informed neural networks? 

ANSWER:
 Physics-informed neural networks (PINNs) are used to solve Partial Differential Equations (PDEs) by embedding physical laws into neural network training [3]. One application of PINNs is in training them to predict solutions for viscous Burgers' equation [1]. PINNs can suffer from unstable convergence, training plateaus, and sensitivity to hyperparameters due to their complex loss function structure [3]. Evolutionary algorithms can be used for hyperparameter optimization in PINNs, combining exploration and exploitation in a two-stage approach to improve accuracy and robustness [3]. This approach involves an outer-loop hyperparameter search that can be treated as a noisy, black-box optimization problem, where evolutionary algorithms are beneficial due to their population-based exploration and ability to handle diverse search spaces [3]. Classical local or gradient-based strategies may get stuck in suboptimal regions for 

In [24]:
# ============================================================
# STAGE 4 — Evaluation Harness (LLM-as-judge + prompt comparison)
# ============================================================
# Goal: stop eyeballing outputs. MEASURE extraction quality on a fixed test
# set, and compare two prompt variants to see which is better. This produces
# a real, defensible metric — the kind you can put on a CV and explain.
#
# DEFEND THIS: an eval harness is how you know a change HELPED instead of
# guessing. Fixed test set + objective scoring + variant comparison = the
# scientific method applied to prompt engineering.

# --- The test set: a few papers we'll evaluate extraction on ---
# DEFEND THIS: a FIXED set means results are comparable across prompt variants
# and reproducible across runs. We use indices into our existing corpus.
TEST_INDICES = [0, 5, 10, 15, 20]
test_set = [papers[i] for i in TEST_INDICES]
print(f"Test set: {len(test_set)} papers")


# --- Prompt Variant A: the original (from Stage 2) ---
def prompt_variant_A(abstract: str) -> str:
    return f"""You are a research assistant analyzing a scientific paper abstract.
Extract the following from the abstract below. If a field is not stated, write "not stated".

Return ONLY a valid JSON object with exactly these keys:
- "method": the main approach or technique used (1-2 sentences)
- "results": the key findings or outcomes (1-2 sentences)
- "limitations": any stated limitations or caveats (1-2 sentences)

Do not include any text outside the JSON object.

ABSTRACT:
{abstract}"""


# --- Prompt Variant B: more structured/explicit (our hypothesis: better) ---
# DEFEND THIS: B adds explicit instructions — be specific, quote where possible,
# distinguish stated-vs-absent. The eval will tell us if this actually helps.
def prompt_variant_B(abstract: str) -> str:
    return f"""You are an expert research analyst extracting structured information from a paper abstract.

Carefully read the abstract and extract these three fields. Be specific and faithful to the text — do not infer beyond what is written. If a field is genuinely not present in the abstract, write exactly "not stated".

Return ONLY a valid JSON object with exactly these keys:
- "method": the specific technique, model, or approach the authors used
- "results": the concrete findings, outcomes, or performance reported
- "limitations": any limitation, caveat, or weakness the authors explicitly mention

Output the JSON object only, with no surrounding text or markdown fences.

ABSTRACT:
{abstract}"""


print("Variants A and B defined.")

Test set: 5 papers
Variants A and B defined.


In [25]:
import json, time

# --- Generic extraction using a given prompt function ---
def run_extraction(abstract: str, prompt_fn) -> dict:
    response = model_llm.generate_content(prompt_fn(abstract))
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    try:
        return {"ok": True, "data": json.loads(raw)}
    except json.JSONDecodeError:
        return {"ok": False, "data": raw}


# --- SCORER 1: structural gate (fast, free, objective) ---
# DEFEND THIS: cheap first check — is it valid JSON with all 3 fields present
# and non-empty? Catches malformed output before we spend an LLM call judging it.
def structural_score(result: dict) -> bool:
    if not result["ok"]:
        return False
    d = result["data"]
    required = ["method", "results", "limitations"]
    return all(k in d and isinstance(d[k], str) and len(d[k].strip()) > 0 for k in required)


# --- SCORER 2: LLM-as-judge (semantic quality) ---
# DEFEND THIS: this is the industry technique — use an LLM to grade whether the
# extraction faithfully reflects the abstract. Returns a 1-5 score. Caveat
# (state this in interviews): the judge is approximate, not ground truth.
def llm_judge_score(abstract: str, extraction: dict) -> int:
    judge_prompt = f"""You are grading the quality of an information extraction.

Given the ABSTRACT and the EXTRACTION (method/results/limitations), rate how accurately and faithfully the extraction reflects the abstract.

Score on a 1-5 scale:
5 = fully accurate and complete
4 = accurate, minor omission
3 = mostly accurate, some issues
2 = partially accurate, notable errors
1 = inaccurate or fabricated

Return ONLY the single integer score (1-5), nothing else.

ABSTRACT:
{abstract}

EXTRACTION:
{json.dumps(extraction)}"""
    resp = model_llm.generate_content(judge_prompt)
    txt = resp.text.strip()
    # pull the first digit 1-5 we find (defensive parsing)
    for ch in txt:
        if ch in "12345":
            return int(ch)
    return 0   # unparseable judge output


print("Runner + scorers ready.")

Runner + scorers ready.


In [27]:
def evaluate_variant(name, prompt_fn):
    struct_passes = 0
    judge_scores = []
    print(f"\n--- Evaluating Variant {name} ---")
    for i, paper in enumerate(test_set, 1):
        result = run_extraction(paper["abstract"], prompt_fn)
        time.sleep(13)   # ~5 req/min limit -> wait between calls
        passed = structural_score(result)
        struct_passes += int(passed)
        if passed:
            score = llm_judge_score(paper["abstract"], result["data"])
            time.sleep(13)
            judge_scores.append(score)
            print(f"  [{i}] structural=PASS  judge={score}/5")
        else:
            judge_scores.append(0)
            print(f"  [{i}] structural=FAIL  judge=0/5")

    n = len(test_set)
    return {"structural_pass_rate": 100*struct_passes/n,
            "avg_judge_score": sum(judge_scores)/n}

results_A = evaluate_variant("A (original)", prompt_variant_A)
results_B = evaluate_variant("B (refined)", prompt_variant_B)

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"Variant A:  structural pass = {results_A['structural_pass_rate']:.0f}%   avg judge = {results_A['avg_judge_score']:.2f}/5")
print(f"Variant B:  structural pass = {results_B['structural_pass_rate']:.0f}%   avg judge = {results_B['avg_judge_score']:.2f}/5")
print("="*50)


--- Evaluating Variant A (original) ---


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 55.961262486s.